# Minería de Datos — Semana 2, Clase 2
## Laboratorio: Preprocesamiento de Datos con scikit-learn

**Institución:** Politécnico Malvinas Argentinas
**Tecnicatura en Informática — 2026**

---

### Objetivos del laboratorio
- **APLICAR** LabelEncoder y OneHotEncoder a variables categóricas del Titanic
- **COMPARAR** StandardScaler vs MinMaxScaler y visualizar el efecto sobre la distribución
- **CONSTRUIR** un pipeline completo con ColumnTransformer, imputación, encoding y Logistic Regression
- **VERIFICAR** la ausencia de data leakage inspeccionando los estadísticos del pipeline
- **EVALUAR** el modelo con accuracy, classification_report y matriz de confusión

### Flujo del laboratorio
1. Setup e importaciones
2. Carga y diagnóstico del dataset Titanic
3. Definir features y hacer train_test_split
4. Demo: Encoding (LabelEncoder + OneHotEncoder)
5. Demo: Scaling (StandardScaler vs MinMaxScaler)
6. Construir Pipeline completo
7. Evaluar resultados
8. Verificar ausencia de leakage
9. ☑ Actividades guiadas (TODO)
10. 🏆 Actividad autónoma


## Celda 1 — Importar librerías y verificar entorno

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, LabelEncoder
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, accuracy_score
)
import warnings
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente ✓")
print(f"  pandas   {pd.__version__}")
print(f"  sklearn  ", end="")
import sklearn; print(sklearn.__version__)


Librerías importadas correctamente ✓
  pandas   2.2.2
  sklearn  1.6.1


## Celda 2 — Cargar el dataset Titanic

Usamos `seaborn.load_dataset('titanic')` para cargar el dataset directamente sin necesidad de subir archivos.
El dataset tiene 891 filas y 15 columnas. Vamos a usar solo un subconjunto.

In [ ]:
# Cargar Titanic desde seaborn (no se necesita subir archivos)
df_raw = sns.load_dataset('titanic')

print(f"Shape original: {df_raw.shape}")
print(f"Columnas: {list(df_raw.columns)}")
print()
df_raw.head(3)


## Celda 3 — Diagnóstico: tipos y valores nulos

Antes de cualquier transformación, necesitamos entender los datos:
- ¿Qué tipo tiene cada columna?
- ¿Cuántos nulos hay?
- ¿Hay categorías con cardinalidad alta?

In [ ]:
# Tipos de datos
print("Tipos de datos:")
print(df_raw.dtypes)
print()

# Porcentaje de nulos por columna
nulos = df_raw.isnull().sum()
pct_nulos = (nulos / len(df_raw) * 100).round(1)
diagnostico = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': pct_nulos
}).sort_values('Porcentaje (%)', ascending=False)
print("Diagnóstico de valores faltantes:")
print(diagnostico[diagnostico['Nulos'] > 0])


> **Observación:** `deck` (equivalente a Cabin) tiene ~77% de nulos → la descartamos.  
> `age` tiene ~20% → imputamos con la mediana.  
> `embarked` tiene 2 nulos → imputamos con el valor más frecuente.

## Celda 4 — Heatmap de correlaciones y distribuciones de variables clave

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribución de edad
axes[0].hist(df_raw['age'].dropna(), bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de Edad')
axes[0].set_xlabel('Edad')
axes[0].axvline(df_raw['age'].median(), color='red', linestyle='--', label=f'Mediana: {df_raw["age"].median():.1f}')
axes[0].legend()

# Supervivencia por sexo
survived_sex = df_raw.groupby('sex')['survived'].mean()
axes[1].bar(survived_sex.index, survived_sex.values, color=['coral','steelblue'])
axes[1].set_title('Tasa de Supervivencia por Sexo')
axes[1].set_ylabel('Proporción de supervivientes')
axes[1].set_ylim(0, 1)

# Supervivencia por clase
survived_pclass = df_raw.groupby('pclass')['survived'].mean()
axes[2].bar(survived_pclass.index.astype(str), survived_pclass.values, color=['gold','silver','peru'])
axes[2].set_title('Tasa de Supervivencia por Clase')
axes[2].set_ylabel('Proporción de supervivientes')
axes[2].set_xlabel('Clase (1=Primera, 3=Tercera)')

plt.tight_layout()
plt.show()


## Celda 5 — Seleccionar columnas y hacer train_test_split

**⚠ REGLA FUNDAMENTAL:** El `train_test_split` se hace ANTES de cualquier preprocesamiento.
Nunca calcular estadísticas (medias, medianas, modas) con el dataset completo antes de dividir.

In [ ]:
# Seleccionar columnas útiles
# Descartamos: name, ticket, cabin/deck (77% nulos), embarked_town, alive, alone, adult_male, who, class
cols_usar = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
df = df_raw[cols_usar].copy()

print(f"Dataset reducido: {df.shape}")
print(f"Columnas: {list(df.columns)}")
print()

# Separar features y target
X = df.drop('survived', axis=1)
y = df['survived']

# PRIMERO: train_test_split con stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # preservar proporción de clases
)

print(f"X_train: {X_train.shape}   X_test: {X_test.shape}")
print(f"Proporción survived en train: {y_train.mean():.3f}")
print(f"Proporción survived en test:  {y_test.mean():.3f}")
print("(Deben ser similares gracias a stratify=y)")


## Celda 6 — Demo: LabelEncoder en columna 'sex'

Primero hacemos esto manualmente para entender cómo funciona.
Luego el pipeline lo va a hacer automáticamente.

In [ ]:
# LabelEncoder para variable binaria 'sex'
le = LabelEncoder()

# Fit SOLO con X_train
le.fit(X_train['sex'])

print("Clases aprendidas:", le.classes_)
print("→ female=0, male=1 (orden alfabético)")
print()

# Ver la transformación
sex_encoded = le.transform(X_train['sex'])
print("Primeros 10 valores originales:", X_train['sex'].values[:10])
print("Primeros 10 valores codificados:", sex_encoded[:10])
print()

# Conteo
print("Distribución:")
print(pd.Series(sex_encoded).value_counts().rename({0:'female (0)', 1:'male (1)'}))


## Celda 7 — Demo: OneHotEncoder en columna 'embarked'

`embarked` tiene 3 categorías: C (Cherbourg), Q (Queenstown), S (Southampton).
Con `drop='first'` generamos solo 2 columnas (evitando multicolinealidad).

In [ ]:
# OneHotEncoder para 'embarked'
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# Primero imputar los 2 nulos (moda = 'S')
embarked_train = X_train[['embarked']].fillna('S')

# Fit SOLO con X_train
ohe.fit(embarked_train)

print("Categorías aprendidas:", ohe.categories_)
print("Columnas generadas (drop='first'):", ohe.get_feature_names_out())
print()

# Transformar
embarked_enc = ohe.transform(embarked_train)
print("Primeras 8 filas (columnas: embarked_Q, embarked_S):")
print(pd.DataFrame(embarked_enc,
                   columns=ohe.get_feature_names_out()).head(8))
print()
print("Interpretación:")
print("  Q=0, S=0 → embarcó en C (Cherbourg) — implícito")
print("  Q=1, S=0 → embarcó en Q (Queenstown)")
print("  Q=0, S=1 → embarcó en S (Southampton)")


## Celda 8 — Demo: StandardScaler vs MinMaxScaler

Comparamos los dos scalers principales y visualizamos su efecto sobre la distribución de `age`.

In [ ]:
# Extraer columna age (sin NaN para el demo manual)
age_train = X_train['age'].copy()
# Para el demo manual, imputamos con la mediana
age_train_filled = age_train.fillna(age_train.median())

# StandardScaler
ss = StandardScaler()
age_std = ss.fit_transform(age_train_filled.values.reshape(-1, 1)).flatten()

# MinMaxScaler
mm = MinMaxScaler()
age_mm = mm.fit_transform(age_train_filled.values.reshape(-1, 1)).flatten()

print(f"Age original  → media: {age_train_filled.mean():.2f}, std: {age_train_filled.std():.2f}")
print(f"StandardScaler→ media: {age_std.mean():.4f}, std: {age_std.std():.4f}")
print(f"MinMaxScaler  → min:   {age_mm.min():.4f}, max: {age_mm.max():.4f}")
print()

# Visualización comparativa
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(age_train_filled, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Age original')
axes[0].set_xlabel('Años')
axes[0].axvline(age_train_filled.mean(), color='red', linestyle='--', label='Media')
axes[0].legend()

axes[1].hist(age_std, bins=30, color='teal', edgecolor='white')
axes[1].set_title('Age — StandardScaler')
axes[1].set_xlabel('z-score (media=0, std=1)')
axes[1].axvline(0, color='red', linestyle='--', label='Media=0')
axes[1].legend()

axes[2].hist(age_mm, bins=30, color='coral', edgecolor='white')
axes[2].set_title('Age — MinMaxScaler')
axes[2].set_xlabel('Valor escalado [0, 1]')
axes[2].axvline(0.5, color='red', linestyle='--', label='0.5')
axes[2].legend()

plt.suptitle('Comparativa: misma distribución, distinta escala', fontsize=13)
plt.tight_layout()
plt.show()


## Celda 9 — Construir el Pipeline completo

Ahora ensamblamos todo en un pipeline que:
1. Imputa y escala las columnas numéricas
2. Imputa y codifica las columnas categóricas
3. Entrena una Regresión Logística

In [ ]:
# Definir qué columnas van a cada transformador
num_cols = ['age', 'fare', 'sibsp', 'parch']
cat_cols = ['sex', 'embarked', 'pclass']

# Pipeline para columnas NUMÉRICAS: imputar → escalar
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Pipeline para columnas CATEGÓRICAS: imputar → OneHotEncoder
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

# Combinar ambos con ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

# Pipeline final: preprocesador + modelo
pipeline = Pipeline([
    ('prep',  preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

print("Pipeline construido:")
print(pipeline)


## Celda 10 — Entrenar el pipeline y evaluar resultados

Una sola llamada `.fit()` ejecuta todos los pasos en orden — imputación, encoding, scaling y entrenamiento del modelo.

In [ ]:
# Entrenar el pipeline completo (solo con X_train)
pipeline.fit(X_train, y_train)
print("Pipeline entrenado ✓")
print()

# Accuracy en train y test
acc_train = pipeline.score(X_train, y_train)
acc_test  = pipeline.score(X_test,  y_test)
print(f"Accuracy en TRAIN: {acc_train:.4f}  ({acc_train*100:.1f}%)")
print(f"Accuracy en TEST:  {acc_test:.4f}  ({acc_test*100:.1f}%)")
print()

# Reporte detallado
y_pred = pipeline.predict(X_test)
print("Classification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=['No sobrevivió (0)', 'Sobrevivió (1)']
))


## Celda 11 — Visualizar: Matriz de confusión y distribución de probabilidades

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Matriz de confusión
ConfusionMatrixDisplay.from_estimator(
    pipeline, X_test, y_test,
    display_labels=['No sobrevivió', 'Sobrevivió'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Matriz de Confusión')

# Distribución de probabilidades predichas
probs = pipeline.predict_proba(X_test)[:, 1]
axes[1].hist(probs[y_test == 0], bins=20, alpha=0.65,
             color='steelblue', label='No sobrevivió (real)')
axes[1].hist(probs[y_test == 1], bins=20, alpha=0.65,
             color='coral', label='Sobrevivió (real)')
axes[1].axvline(0.5, color='black', linestyle='--', label='Umbral 0.5')
axes[1].set_xlabel('Probabilidad predicha de supervivencia')
axes[1].set_ylabel('Cantidad de pasajeros')
axes[1].set_title('Distribución de probabilidades predichas')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Total predicciones correctas:   {(y_pred == y_test).sum()} / {len(y_test)}")
print(f"Falsos positivos (FP): predijo sobrevivió, no sobrevivió: {((y_pred==1)&(y_test==0)).sum()}")
print(f"Falsos negativos (FN): predijo no, sí sobrevivió:         {((y_pred==0)&(y_test==1)).sum()}")


## Celda 12 — Verificar que no hay Data Leakage

Inspeccionamos los estadísticos que el pipeline aprendió y confirmamos que son los del conjunto de entrenamiento.

In [ ]:
# Acceder a los componentes internos del pipeline
preprocessor_fit = pipeline.named_steps['prep']
num_transformer  = preprocessor_fit.named_transformers_['num']
imputer          = num_transformer.named_steps['imputer']
scaler           = num_transformer.named_steps['scaler']

print("=== Verificación de estadísticos aprendidos ===")
print()
print(f"Columnas numéricas: {num_cols}")
print()
print(f"Medianas aprendidas por el imputador: {imputer.statistics_.round(3)}")
print(f"Medianas reales de X_train:           {X_train[num_cols].median().values.round(3)}")
print()
print(f"Medias aprendidas por el scaler:      {scaler.mean_.round(3)}")
print(f"Medias reales de X_train:             {X_train[num_cols].mean().values.round(3)}")
print()
# Comparar con las del dataset completo (para demostrar que son distintas)
print("Comparación con el dataset COMPLETO (evidencia de no-leakage):")
print(f"Mediana de 'age' en X_train: {X_train['age'].median():.2f}")
print(f"Mediana de 'age' en df total: {df['age'].median():.2f}")
print()
if abs(X_train['age'].median() - df['age'].median()) > 0.1:
    print("✓ Son distintas → el pipeline aprendió solo de X_train (sin leakage)")
else:
    print("⚠ Son muy similares en este caso (dataset balanceado), pero el proceso es correcto")


---
## ☑ Actividades Guiadas — TODO

Estas celdas tienen `# TODO: ` para que completes el código.
Las soluciones están comentadas más abajo — ¡intentalo solo primero!

---
### Actividad 1 ★ Básico — Comparar estrategias de imputación en 'age'

Comparar `strategy='mean'` vs `strategy='median'` y ver la diferencia en la distribución.

In [ ]:
# TODO 1: Crear dos imputadores y aplicarlos a X_train['age']
# Luego graficar los tres histogramas: original, imputado con mean, imputado con median

# 1a. Crear imp_mean y imp_median
# TODO: imp_mean = SimpleImputer(strategy=???)
# TODO: imp_median = SimpleImputer(strategy=???)

# 1b. Hacer fit SOLO en X_train[['age']]
# TODO: imp_mean.fit(???)
# TODO: imp_median.fit(???)

# 1c. Transformar
# TODO: age_mean   = imp_mean.transform(X_train[['age']]).flatten()
# TODO: age_median = imp_median.transform(X_train[['age']]).flatten()

# 1d. Graficar
# TODO: Crear fig con 3 subplots y comparar las distribuciones

# ─── SOLUCIÓN (descomentar para ver) ───────────────────────────────────────
# imp_mean   = SimpleImputer(strategy='mean')
# imp_median = SimpleImputer(strategy='median')
# imp_mean.fit(X_train[['age']])
# imp_median.fit(X_train[['age']])
# age_mean   = imp_mean.transform(X_train[['age']]).flatten()
# age_median = imp_median.transform(X_train[['age']]).flatten()
#
# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# axes[0].hist(X_train['age'].dropna(), bins=30, color='steelblue')
# axes[0].set_title(f'Age original (NaN: {X_train["age"].isna().sum()})')
# axes[1].hist(age_mean,   bins=30, color='coral')
# axes[1].set_title(f'Imputado con mean ({imp_mean.statistics_[0]:.1f})')
# axes[2].hist(age_median, bins=30, color='teal')
# axes[2].set_title(f'Imputado con median ({imp_median.statistics_[0]:.1f})')
# plt.suptitle('Efecto de la estrategia de imputación sobre la distribución')
# plt.tight_layout(); plt.show()
#
# print(f"Media del imputador  mean:   {imp_mean.statistics_[0]:.2f}")
# print(f"Media del imputador  median: {imp_median.statistics_[0]:.2f}")
# print("Observación: ¿se notan los 'picos' artificiales en la distribución?")


### Actividad 2 ★★ Medio — Comparar StandardScaler vs MinMaxScaler en el pipeline completo

In [ ]:
# TODO 2: Crear dos versiones del pipeline y comparar accuracy

# 2a. pipeline_std: usar StandardScaler (ya lo tenemos arriba)
# pipeline_std = pipeline  # el que ya construimos

# 2b. Crear pipeline_mm con MinMaxScaler en num_pipe
# TODO: num_pipe_mm = Pipeline([
#     ('imputer', SimpleImputer(strategy='median')),
#     ('scaler',  MinMaxScaler())
# ])

# 2c. Crear preprocessor_mm y pipeline_mm completo
# TODO: preprocessor_mm = ColumnTransformer([...])
# TODO: pipeline_mm = Pipeline([...])

# 2d. Entrenar ambos y comparar accuracy en X_test
# TODO: pipeline_mm.fit(X_train, y_train)
# TODO: acc_std = pipeline.score(X_test, y_test)
# TODO: acc_mm  = pipeline_mm.score(X_test, y_test)
# TODO: print(f"Accuracy StandardScaler: {acc_std:.4f}")
# TODO: print(f"Accuracy MinMaxScaler:   {acc_mm:.4f}")

# ─── SOLUCIÓN ──────────────────────────────────────────────────────────────
# num_pipe_mm = Pipeline([
#     ('imputer', SimpleImputer(strategy='median')),
#     ('scaler',  MinMaxScaler())
# ])
# preprocessor_mm = ColumnTransformer([
#     ('num', num_pipe_mm, num_cols),
#     ('cat', cat_pipe,    cat_cols)
# ])
# pipeline_mm = Pipeline([
#     ('prep',  preprocessor_mm),
#     ('model', LogisticRegression(max_iter=1000, random_state=42))
# ])
# pipeline_mm.fit(X_train, y_train)
# acc_std = pipeline.score(X_test, y_test)
# acc_mm  = pipeline_mm.score(X_test, y_test)
# print(f"Accuracy con StandardScaler: {acc_std:.4f}")
# print(f"Accuracy con MinMaxScaler:   {acc_mm:.4f}")
# print()
# print("Observación: para Logistic Regression, ambos scalers suelen dar")
# print("resultados muy similares. La diferencia es mayor en KNN o SVM.")


### Actividad 3 ★★ Medio — Reemplazar SimpleImputer por KNNImputer

In [ ]:
# TODO 3: Crear pipeline_knn con KNNImputer en lugar de SimpleImputer para numéricas

# 3a. Crear num_pipe_knn con KNNImputer(n_neighbors=5)
# TODO: num_pipe_knn = Pipeline([
#     ('imputer', KNNImputer(n_neighbors=5)),
#     ('scaler',  StandardScaler())
# ])

# 3b. Crear pipeline_knn completo
# Nota: KNNImputer NO acepta columnas categóricas → solo usarlo en num_cols

# 3c. Entrenar y comparar con el pipeline original
# TODO: pipeline_knn.fit(X_train, y_train)
# TODO: Comparar accuracy Y classification_report

# ─── SOLUCIÓN ──────────────────────────────────────────────────────────────
# num_pipe_knn = Pipeline([
#     ('imputer', KNNImputer(n_neighbors=5)),
#     ('scaler',  StandardScaler())
# ])
# preprocessor_knn = ColumnTransformer([
#     ('num', num_pipe_knn, num_cols),
#     ('cat', cat_pipe,     cat_cols)
# ])
# pipeline_knn = Pipeline([
#     ('prep',  preprocessor_knn),
#     ('model', LogisticRegression(max_iter=1000, random_state=42))
# ])
# pipeline_knn.fit(X_train, y_train)
# acc_original = pipeline.score(X_test, y_test)
# acc_knn      = pipeline_knn.score(X_test, y_test)
# print(f"Accuracy SimpleImputer: {acc_original:.4f}")
# print(f"Accuracy KNNImputer:    {acc_knn:.4f}")
# print()
# y_pred_knn = pipeline_knn.predict(X_test)
# print("Reporte con KNNImputer:")
# print(classification_report(y_test, y_pred_knn,
#       target_names=['No sobrevivió', 'Sobrevivió']))
# print("Observación: KNNImputer suele mejorar marginalmente (~1-2%)")
# print("a costa de un tiempo de entrenamiento mayor.")


---
## 🏆 Actividad Autónoma — Pipeline desde cero con otro dataset

Elegí **uno** de los siguientes datasets y construí un pipeline completo desde cero.

| Dataset | Carga | Target sugerido | Dificultad |
|---------|-------|-----------------|------------|
| Tips | `sns.load_dataset('tips')` | `tip > 20% de total` | ★ Fácil |
| Penguins | `sns.load_dataset('penguins')` | `species` (3 clases) | ★★ Medio |
| Diamonds | `sns.load_dataset('diamonds')` | `price > mediana` | ★★ Medio |

### Tu pipeline debe incluir obligatoriamente:
1. EDA mínimo: shape, dtypes, nulos, distribuciones
2. Definir X e y, y hacer `train_test_split` con `stratify=y`
3. Identificar columnas numéricas y categóricas
4. `num_pipe`: SimpleImputer + StandardScaler (o RobustScaler si hay outliers)
5. `cat_pipe`: SimpleImputer + OneHotEncoder(drop='first')
6. `ColumnTransformer` + `Pipeline` con `LogisticRegression`
7. Evaluar con `accuracy_score` Y `classification_report`
8. Comentar: ¿qué mejorarías para obtener mejor accuracy?


In [ ]:
# ─── Tu pipeline autónomo ────────────────────────────────────────────────────

# 1. Cargar el dataset elegido
# df_auto = sns.load_dataset('???')  # tips / penguins / diamonds
# print(df_auto.shape)
# df_auto.head()

# 2. EDA mínimo
# ...

# 3. Definir X e y + train_test_split
# X_auto = ...
# y_auto = ...
# X_tr, X_te, y_tr, y_te = train_test_split(X_auto, y_auto, ...)

# 4. Identificar columnas
# num_cols_auto = [...]
# cat_cols_auto = [...]

# 5. Construir num_pipe y cat_pipe
# num_pipe_auto = Pipeline([...])
# cat_pipe_auto = Pipeline([...])

# 6. ColumnTransformer + Pipeline
# preprocessor_auto = ColumnTransformer([...])
# pipeline_auto = Pipeline([...])

# 7. Entrenar y evaluar
# pipeline_auto.fit(X_tr, y_tr)
# print(f"Accuracy: {pipeline_auto.score(X_te, y_te):.4f}")
# y_pred_auto = pipeline_auto.predict(X_te)
# print(classification_report(y_te, y_pred_auto))

# 8. Conclusion: que mejorarias?
# '''
# Respuesta aqui...
# '''


---
## 📊 Resumen de lo que construimos hoy

| Paso | Herramienta | Qué hace |
|------|-------------|----------|
| Diagnóstico | `.info()`, `.isnull()` | Entender el estado del dataset |
| Split | `train_test_split(stratify=y)` | Dividir ANTES de cualquier transformación |
| Imputar numéricas | `SimpleImputer(strategy='median')` | Reemplazar NaN con la mediana |
| Escalar | `StandardScaler()` | Media=0, Std=1 |
| Imputar categóricas | `SimpleImputer(strategy='most_frequent')` | Reemplazar NaN con la moda |
| Codificar | `OneHotEncoder(drop='first')` | N categorías -> N-1 columnas binarias |
| Combinar | `ColumnTransformer([('num',...),('cat',...)])` | Aplicar distintos pipes por columna |
| Pipeline final | `Pipeline([('prep',...),('model',...)])` | Encadenar todo sin leakage |
| Evaluar | `accuracy`, `classification_report` | Métricas honestas sobre test set |

**Resultado esperado:** accuracy ~79-81% con Logistic Regression sobre Titanic.

---
*Mineria de Datos - Semana 2, Clase 2 | Politecnico Malvinas Argentinas | 2026*
